# Day 2: Spatial Clustering & Hotspot Identification
## K-Means vs. DBSCAN with Haversine Spherical Metric

### Theoretical Motivation:
Accident hotspots are geographic concentrations of historical accidents.
1. **The Haversine Metric:** Latitude and longitude are angular coordinates on a sphere. Treating them as flat Euclidean $(x, y)$ coordinates introduces distortion. The Haversine formula calculates the true great-circle distance:
$$d = 2R \arcsin\left(\sqrt{\sin^2\left(\frac{\Delta \phi}{2}\right) + \cos(\phi_1)\cos(\phi_2)\sin^2\left(\frac{\Delta \lambda}{2}\right)}\right)$$
2. **K-Means vs DBSCAN:**
- **K-Means** requires specifying $K$ upfront and forces all points (including remote rural accidents) into spherical clusters.
- **DBSCAN** discovers clusters of arbitrary geometric shape (e.g. winding freeways) and labels isolated events as noise.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans, DBSCAN
from pathlib import Path

EARTH_RADIUS_KM = 6371.0088

# Load California subset
ca_path = Path('../data/processed/california_accidents.csv')
if not ca_path.exists():
    ca_path = Path('data/processed/california_accidents.csv')

df_ca = pd.read_csv(ca_path)
print(f"Loaded {len(df_ca):,} California accident records for clustering.")

In [ ]:
# 1. Evaluate K-Means Inertia for Multiple K
coords = df_ca[['Start_Lat', 'Start_Lng']].values[:15000]

k_range = [5, 8, 10, 15, 20]
inertias = []

for k in k_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    km.fit(coords)
    inertias.append(km.inertia_)
    print(f"K = {k:2d} -> Inertia: {km.inertia_:.2f}")

plt.figure(figsize=(7, 4))
plt.plot(k_range, inertias, marker='s', color='#6366f1', linewidth=2)
plt.title('K-Means Elbow Curve (Inertia vs K)')
plt.xlabel('Number of Clusters (K)')
plt.ylabel('Inertia (Sum of Squared Distances)')
plt.show()

In [ ]:
# 2. DBSCAN Clustering with Spherical Haversine Metric
# Convert coordinates to radians
coords_rad = np.radians(coords)

# Configure DBSCAN: eps = 7.5 km in radians, min_samples = 35
eps_km = 7.5
eps_rad = eps_km / EARTH_RADIUS_KM
min_samples = 35

dbscan = DBSCAN(eps=eps_rad, min_samples=min_samples, metric='haversine', algorithm='ball_tree')
db_labels = dbscan.fit_predict(coords_rad)

n_clusters = len(set(db_labels) - {-1})
n_noise = np.sum(db_labels == -1)

print(f"DBSCAN Results:")
print(f"- Dense Clusters Identified: {n_clusters}")
print(f"- Noise / Outlier Points: {n_noise} ({n_noise / len(coords) * 100:.2f}%)")

In [ ]:
# 3. Visualize Spatial Clusters (California)
plt.figure(figsize=(9, 7))
plt.scatter(coords[db_labels == -1, 1], coords[db_labels == -1, 0], c='gray', s=5, alpha=0.2, label='Noise (Isolated)')
plt.scatter(coords[db_labels != -1, 1], coords[db_labels != -1, 0], c=db_labels[db_labels != -1], cmap='tab10', s=12, alpha=0.7)
plt.title('DBSCAN Haversine Clustering: California Accident Hotspots')
plt.xlabel('Longitude')
plt.ylabel('Latitude')
plt.legend()
plt.show()